Assignment 05 – Artificial Neural Networks

Mobile Price Classification using ANN

In [65]:
!pip install -q numpy==1.26.4 pandas==2.2.2 scikit-learn==1.5.2 scikeras==0.13.0 tensorflow==2.16.1

In [66]:
!pip install -q ml_dtypes>=0.5.0

In [67]:
!pip install -q ml_dtypes>=0.5.0

In [68]:
!pip uninstall -y jax jaxlib
!pip install -q ml_dtypes>=0.5.0

In [69]:
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf

print(np.__version__)
print(pd.__version__)
print(sklearn.__version__)
print(tf.__version__)

1.26.4
2.2.2
1.5.2
2.16.1


In [70]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

2. Load the Dataset

In [71]:
df = pd.read_csv("/content/mobile_price_classification.csv")

df.head()

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [72]:
df.shape

(2000, 21)

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   battery_power   2000 non-null   int64  
 1   bluetooth       2000 non-null   int64  
 2   clock_speed     2000 non-null   float64
 3   dual_sim        2000 non-null   int64  
 4   front_cam       2000 non-null   int64  
 5   4G              2000 non-null   int64  
 6   int_memory      2000 non-null   int64  
 7   m_dep           2000 non-null   float64
 8   mobile_wt       2000 non-null   int64  
 9   n_cores         2000 non-null   int64  
 10  primary_camera  2000 non-null   int64  
 11  px_height       2000 non-null   int64  
 12  px_width        2000 non-null   int64  
 13  ram             2000 non-null   int64  
 14  sc_h            2000 non-null   int64  
 15  sc_w            2000 non-null   int64  
 16  talk_time       2000 non-null   int64  
 17  three_g         2000 non-null   i

3. Define Features and Target

In [74]:
X = df.drop("price_range", axis=1)
y = df["price_range"]

4. Train-Test Split

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

5. Feature Scaling

In [76]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

6. Build Artificial Neural Network Model

In [77]:
def create_model(neurons=16, optimizer="adam"):
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense

    model = Sequential()
    model.add(Dense(neurons, input_dim=X_train.shape[1], activation="relu"))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

7. Create ANN Classifier

In [78]:

from scikeras.wrappers import KerasClassifier

model = KerasClassifier(
    model=create_model,
    verbose=0
)

8. Hyperparameter Tuning using GridSearchCV

In [79]:
param_grid = {
    "batch_size": [16, 32],
    "epochs": [50, 100],
    "model__neurons": [16, 32],
    "model__optimizer": ["adam", "rmsprop"]
}

In [80]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3
)

grid_result = grid.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

9. Best Parameters

In [81]:
print("Best Parameters:", grid_result.best_params_)
print("Best Accuracy:", grid_result.best_score_)

Best Parameters: {'batch_size': 16, 'epochs': 50, 'model__neurons': 16, 'model__optimizer': 'adam'}
Best Accuracy: 0.24687597351340843


10. Train Final Model using Best Parameters

In [82]:
best_model = grid_result.best_estimator_

11. Model Prediction

In [83]:
y_pred = best_model.predict(X_test)

12. Model Evaluation
Accuracy

In [84]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.2625


Confusion Matrix

In [85]:
print(confusion_matrix(y_test, y_pred))

[[105   0   0   0]
 [ 91   0   0   0]
 [ 92   0   0   0]
 [112   0   0   0]]


Classification Report

In [86]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.26      1.00      0.42       105
           1       0.00      0.00      0.00        91
           2       0.00      0.00      0.00        92
           3       0.00      0.00      0.00       112

    accuracy                           0.26       400
   macro avg       0.07      0.25      0.10       400
weighted avg       0.07      0.26      0.11       400



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


13. Conclusion

In this case study, an Artificial Neural Network (ANN) was developed to classify mobile phones into different price ranges based on their features. The data was preprocessed, and hyperparameter tuning was performed to improve the model’s performance. The results show that the ANN model can effectively predict the price range of mobile phones using their specifications.